In [25]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path
import numpy as np
import os
import shutil
import json
import zipfile
import re

sys.path.append(str(Path().resolve().parents[0]))

# Local src

from src.df_overview import df_overview
from src.generate_metadata import generate_metadata

print('Done')

Done


In [26]:
path_rfm_csv = '../data/02_processed/rfm_retail.csv'
path_base_csv = '../data/02_processed/base_retail.csv'

df_rfm = pd.read_csv(path_rfm_csv)
df_base = pd.read_csv(path_base_csv)

In [27]:
df_overview(df_rfm)

================================= Head =================================
   customer_id   last_purchase_date  frequency  monetary  recency  r_score  \
0        17592  2009-12-01 10:49:00          1    148.30    738.0        1   
1        17818  2009-12-02 11:34:00          1    124.78    737.0        1   
2        17087  2009-12-02 10:41:00          1    221.53    737.0        1   
3        17056  2009-12-01 12:55:00          1    128.60    737.0        1   
4        13526  2009-12-01 13:13:00          2   1182.00    737.0        1   

   f_score  m_score  
0        1        1  
1        1        1  
2        1        1  
3        1        1  
4        3        3  
================================= Tail =================================
      customer_id   last_purchase_date  frequency  monetary  recency  r_score  \
5842        12985  2011-12-09 10:46:00          2   1239.38      0.0        5   
5843        14056  2011-12-08 13:36:00         23   8152.71      0.0        5   
5844      

In [28]:
df_overview(df_base)

================================= Head =================================
  invoice stock_code                          description  quantity  \
0  489434      85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434     79323P                   PINK CHERRY LIGHTS        12   
2  489434     79323W                  WHITE CHERRY LIGHTS        12   
3  489434      22041          RECORD FRAME 7" SINGLE SIZE        48   
4  489434      21232       STRAWBERRY CERAMIC TRINKET BOX        24   

          invoice_date  price  customer_id         country  revenue  
0  2009-12-01 07:45:00   6.95        13085  United Kingdom     83.4  
1  2009-12-01 07:45:00   6.75        13085  United Kingdom     81.0  
2  2009-12-01 07:45:00   6.75        13085  United Kingdom     81.0  
3  2009-12-01 07:45:00   2.10        13085  United Kingdom    100.8  
4  2009-12-01 07:45:00   1.25        13085  United Kingdom     30.0  
================================= Tail =================================
       

            quantity          price    customer_id        revenue
count  793755.000000  793755.000000  793755.000000  793755.000000
mean       12.643784       2.969689   15319.991521      20.609825
std       142.083231       4.474045    1693.625884     148.643057
min    -74215.000000       0.001000   12346.000000  -77183.600000
25%         2.000000       1.250000   13971.000000       4.350000
50%         5.000000       1.950000   15245.000000      11.700000
75%        12.000000       3.750000   16791.000000      19.500000
max     74215.000000     649.500000   18287.000000   77183.600000
================================= NaN =================================
invoice         0
stock_code      0
description     0
quantity        0
invoice_date    0
price           0
customer_id     0
country         0
revenue         0
dtype: int64
================================= Duplicates =================================
0
================================= Cardinality & Top Values ===================

In [30]:
df_rfm["recency_score"] = pd.qcut(
    df_rfm["recency"], 
    5, 
    labels=[5,4,3,2,1], 
    duplicates='drop'
)

df_rfm["frequency_score"] = pd.qcut(
    df_rfm["frequency"].rank(method="first"), 
    5, 
    labels=[1,2,3,4,5], 
    duplicates='drop'
)

df_rfm["monetary_score"] = pd.qcut(
    df_rfm["monetary"], 
    5, 
    labels=[1,2,3,4,5], 
    duplicates='drop'
)

df_rfm["recency_score"] = df_rfm["recency_score"].astype(int)
df_rfm["frequency_score"] = df_rfm["frequency_score"].astype(int)
df_rfm["monetary_score"] = df_rfm["monetary_score"].astype(int)

df_rfm["R_F_Score"] = df_rfm["recency_score"].astype(str) + df_rfm["frequency_score"].astype(str)

seg_map = {
    r'[1-2][1-2]': 'hibernating',
    r'[1-2][3-4]': 'at_risk',
    r'[1-2]5': 'cant_lose',
    r'3[1-2]': 'about_to_sleep',
    r'33': 'need_attention',
    r'[3-4][4-5]': 'loyal_customers',
    r'41': 'promising',
    r'51': 'new_customers',
    r'[4-5][2-3]': 'potential_loyalists',
    r'5[4-5]': 'champions'
}

df_rfm["segment"] = df_rfm["R_F_Score"].replace(seg_map, regex = True)

In [31]:
df_base['sales_only'] = df_base['revenue'].clip(lower=0)
df_base['returns_only'] = df_base['revenue'].clip(upper=0).abs()

ratios = df_base.groupby('customer_id').agg(
    total_sales=('sales_only', 'sum'),
    total_returns=('returns_only', 'sum')
)

ratios['return_ratio'] = ratios['total_returns'] / ratios['total_sales']
ratios['return_ratio'] = ratios['return_ratio'].fillna(0)
ratios.reset_index()
ratios
df_rfm = df_rfm.set_index('customer_id')
df_rfm = df_rfm.merge(ratios[['return_ratio']], left_index=True, right_index=True, how='left')

In [32]:
df_overview(df_rfm)

================================= Head =================================
              last_purchase_date  frequency  monetary  recency  r_score  \
customer_id                                                               
17592        2009-12-01 10:49:00          1    148.30    738.0        1   
17818        2009-12-02 11:34:00          1    124.78    737.0        1   
17087        2009-12-02 10:41:00          1    221.53    737.0        1   
17056        2009-12-01 12:55:00          1    128.60    737.0        1   
13526        2009-12-01 13:13:00          2   1182.00    737.0        1   

             f_score  m_score  recency_score  frequency_score  monetary_score  \
customer_id                                                                     
17592              1        1              1                1               1   
17818              1        1              1                1               1   
17087              1        1              1                1               1

In [33]:
pd_to_csv_path = os.path.join('..', 'data', '03_featured', 'featured_rfm_retail.csv')

os.makedirs(os.path.dirname(pd_to_csv_path), exist_ok=True)
df_rfm.to_csv(pd_to_csv_path, index=False)

generate_metadata(
    df=df_rfm,
    csv_path=pd_to_csv_path
)

print(f"Saved to: {pd_to_csv_path}")

Metadata saved: ../data/03_featured/featured_rfm_retail_metadata.json
Saved to: ../data/03_featured/featured_rfm_retail.csv


customer_age_days = snapshot_date - first_purchase_date
avg_interpurchase_time = customer_age_days / (frequency - 1)
avg_order_value